In [1]:
!pip install unsloth
!pip install rouge-score bert-score datasets accelerate trl peft bitsandbytes
!pip install datasets==3.6.0
import datasets
dataset = datasets.load_dataset('csebuetnlp/xlsum','serbian_latin')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:

print(f"Train: {len(dataset['train'])}")
print(f"Valid: {len(dataset['validation'])}")
print(f"Test:  {len(dataset['test'])}")

ex = dataset["train"][0]
print("\nTitle:", ex["title"])
print("Article:", ex["text"][:300], "...")
print("Summary:", ex["summary"])


Train: 7276
Valid: 909
Test:  909

Title: Mirijam Menken: Naučnica koja je stvorila život u epruveti
Article: Priča o Mirijam Menkin malo je drugačija. Jednog utorka u februaru 1944. godine, ova 43-godišnja laboratorijska tehničarka ostala je budna čitavu noć umirujući osmomesečnu ćerku - „uzorak uživo", imala je običaj da kaže za nju - kojoj su upravo počeli da rastu zubi. Narednog jutra, Menkin je otišla  ...
Summary: U klasičnoj priči o naučnom otkriću, istraživač radi do duboko u noć, sam u laboratoriji. Odjednom, na pamet mu pada genijalna pomisao: jabuka mu padne na glavu, munje zvekne ključ, pojave se bakterije u Petrijevoj posudi. I eureka: otkriće!


In [3]:
import re

def clean_text(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
def filter_by_length(example):
    article_words = len(example["text"].split())
    summary_words = len(example["summary"].split())
    return 50 < article_words < 3000 and 10 < summary_words < 300

dataset_filtered = dataset.filter(filter_by_length)
print(f"После фильтрации — Train: {len(dataset_filtered['train'])}, "
      f"Valid: {len(dataset_filtered['validation'])}, "
      f"Test: {len(dataset_filtered['test'])}")

train_subset = dataset_filtered["train"].select(range(min(2000, len(dataset_filtered["train"]))))
val_subset   = dataset_filtered["validation"].select(range(min(100, len(dataset_filtered["validation"]))))
test_subset  = dataset_filtered["test"].select(range(min(100, len(dataset_filtered["validation"]))))

SYSTEM_PROMPT = (
    "Ti si AI asistent specijalizovan za sažimanje novinskih članaka "
    "na srpskom jeziku. Napravi kratak i informativan sažetak datog članka."
)

def format_example(example):
    """Формирует чат-сообщение для SFT"""
    article = clean_text(example["text"])
    summary = clean_text(example["summary"])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Sažmi sledeći članak:\n\n{article}"},
        {"role": "assistant", "content": summary},
    ]
    return {"messages": messages}

train_dataset = train_subset.map(format_example)
val_dataset   = val_subset.map(format_example)
test_dataset  = test_subset.map(format_example)

print(f"Готово: {len(train_dataset)} train, {len(val_dataset)} val, {len(test_dataset)} test")
print("\nПример messages[0]:", train_dataset[0]["messages"][1]["content"][:200])

После фильтрации — Train: 6602, Valid: 887, Test: 894


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Готово: 2000 train, 100 val, 100 test

Пример messages[0]: Sažmi sledeći članak:

Priča o Mirijam Menkin malo je drugačija. Jednog utorka u februaru 1944. godine, ova 43-godišnja laboratorijska tehničarka ostala je budna čitavu noć umirujući osmomesečnu ćerku


In [4]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
)

print(model.print_trainable_parameters())


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301
None


In [5]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen3_8b_serbian_sum",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.05,
    learning_rate=1e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

...

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,772 | Num Epochs = 1 | Total steps = 222
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)


Step,Training Loss,Validation Loss
50,2.076855,1.982436
100,1.988747,1.919367
150,1.982541,1.899832
200,1.951644,1.893983


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=222, training_loss=2.0173344139580256, metrics={'train_runtime': 12324.5664, 'train_samples_per_second': 0.144, 'train_steps_per_second': 0.018, 'total_flos': 1.6537840171723776e+17, 'train_loss': 2.0173344139580256, 'epoch': 1.0})

In [6]:
model.save_pretrained("qwen3_8b_serbian_sum_lora")
tokenizer.save_pretrained("qwen3_8b_serbian_sum_lora")

import shutil
shutil.make_archive("/content/lora_adapter_8b", "zip", "/content/qwen3_8b_serbian_sum_lora")

from google.colab import files
files.download("/content/lora_adapter_8b.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>